Load cleaned dataset

In [9]:
import os
%cd ~/complaint-routing-ml
os.getcwd()

/Users/mikael_abyssinia/complaint-routing-ml


'/Users/mikael_abyssinia/complaint-routing-ml'

In [10]:
import pandas as pd

df = pd.read_parquet("data/processed/complaints_department_model.parquet")

Confirm shape

In [11]:
df.shape

(1404900, 2)

In [12]:
df.head()

,Product,Consumer complaint narrative
0,Credit Reporting,Kindly address this issue on my credit report....
1,Credit Reporting,XXXX XXXX has a old account settled in XXXX th...
2,Credit Reporting,These are not my accounts.
3,Credit Reporting,"I wrote three requests, the unverified account..."
4,Credit Reporting,I was recently going to check out a new car at...


Check complaint length distribution

In [13]:
df["word_count"] = df["Consumer complaint narrative"].str.split().str.len()

In [14]:
df["word_count"].describe()

count    1.404900e+06
mean     1.762166e+02
std      2.220826e+02
min      1.000000e+00
25%      6.000000e+01
50%      1.160000e+02
75%      2.110000e+02
max      6.469000e+03
Name: word_count, dtype: float64

Check how many complaints are extremely short

In [15]:
(df["word_count"] <= 5).sum()
print(((df["word_count"] <= 5).mean() * 100).round(2))

0.13


Check complaint length by department

In [16]:
df.groupby("Product")["word_count"].describe()

,count,mean,std,min,25%,50%,75%,max
Product,,,,,,,,
Bank Accounts,68807.0,220.459067,233.739630,1.0,81.0,157.0,278.0,5813.0
Consumer Loans,15975.0,203.378967,201.620245,1.0,78.0,146.0,258.0,3951.0
Credit Reporting,944120.0,161.962673,213.565656,1.0,56.0,107.0,192.0,6312.0
Credit or Prepaid Cards,86483.0,212.101384,223.742657,1.0,78.0,155.0,270.0,6469.0
Debt collection,152342.0,165.700608,211.918653,1.0,53.0,106.0,204.0,5905.0
Financial Services Support,1833.0,164.931806,187.555230,4.0,43.0,106.0,219.0,2262.0
Money Transfers and Digital Payments,42516.0,172.771921,205.861750,2.0,99.0,105.0,192.0,5988.0
Mortgage,52591.0,297.585690,307.140787,1.0,123.0,214.0,372.0,5882.0
Student loan,22273.0,225.042069,226.359255,1.0,92.0,167.0,282.0,5546.0



Vocabulary Exploration

In [17]:
from collections import Counter
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS)
custom_stop = {"xxxx", "xx"}  # remove redaction tokens
stop_words = stop_words.union(custom_stop)

def get_top_words_clean(df, department, n=20):
    texts = df[df["Product"] == department]["Consumer complaint narrative"]
    
    all_words = []
    for text in texts:
        text = text.lower()
        words = re.findall(r"\b[a-z]+\b", text)
        words = [w for w in words if w not in stop_words and len(w) > 2]
        all_words.extend(words)
    
    return Counter(all_words).most_common(n)


In [18]:
for dept in df["Product"].unique():
    print("\n", dept)
    print(get_top_words_clean(df, dept, 15))


 Credit Reporting
[('credit', 2333401), ('report', 1525639), ('information', 1431905), ('account', 1273188), ('reporting', 1256124), ('consumer', 1195864), ('accounts', 784101), ('section', 488342), ('inaccurate', 485030), ('fcra', 446737), ('act', 397373), ('agency', 387902), ('identity', 377870), ('fair', 371082), ('date', 367254)]

 Student loan
[('loan', 48005), ('loans', 33441), ('payment', 28860), ('payments', 26503), ('student', 21492), ('account', 18194), ('navient', 15846), ('time', 14930), ('told', 14308), ('credit', 13825), ('information', 13545), ('pay', 12179), ('received', 11887), ('did', 10495), ('mohela', 10349)]

 Credit or Prepaid Cards
[('card', 179935), ('credit', 176670), ('account', 146932), ('payment', 69955), ('bank', 68472), ('did', 51422), ('told', 50897), ('received', 49787), ('called', 48521), ('balance', 44324), ('time', 44010), ('information', 38861), ('charge', 36695), ('dispute', 36379), ('late', 35896)]

 Debt collection
[('debt', 289692), ('credit', 2